# FarmTech na Era da Cloud Computing
## Entrega 1 — Machine Learning (`crop_yield.csv`)

**Notebook oficial:** `HigorHenriqueGarcia_rm571820_pbl_fase4.ipynb`  
**Dataset:** `data/crop_yield.csv` (156 registros, alvo real `Yield`)  
**Protocolo comum:** `random_state=42`, teste 25%, Isolation Forest (3%) só no treino.

| Integrante | RM | Parte |
|---|---|---|
| Higor Henrique Garcia | 571820 | Dicionário, limpeza, EDA e regressão linear |
| Vinicius Alves Lopes dos Anjos | 572814 | Clusterização, outliers, Random Forest e Gradient Boosting |
| Humberto | 570536 | Decision Tree e KNN |
| Igor | 572822 | Comparação AWS (Entrega 2 no README) |


## 1. Dicionário e qualidade dos dados

Colunas oficiais: cultura, precipitação (mm/dia), umidade específica (g/kg),
umidade relativa (%), temperatura (°C) e rendimento. Não há target sintético.
Após a limpeza restam 156 linhas; nulos numéricos são imputados por mediana
apenas no pipeline de modelo.


In [1]:
from pathlib import Path
import pandas as pd
from higor_eda import run_eda, save_outputs

eda = run_eda(random_state=42)
save_outputs(eda)
print(eda.column_dictionary.to_string(index=False))
print()
print(eda.crop_counts.to_string(index=False))
print()
print(eda.null_counts.to_string(index=False))


                              column       dtype                                                description
                                Crop categorical     Cultura agrícola registrada no observatório climático.
            Precipitation (mm day-1)     numeric                   Precipitação média diária em milímetros.
Specific Humidity at 2 Meters (g/kg)     numeric                Umidade específica do ar a 2 metros (g/kg).
   Relative Humidity at 2 Meters (%)     numeric                     Umidade relativa do ar a 2 metros (%).
         Temperature at 2 Meters (C)     numeric                   Temperatura média do ar a 2 metros (°C).
                               Yield     numeric Produtividade observada da cultura (alvo real do dataset).

           crop  count
   Cocoa, beans     39
 Oil palm fruit     39
    Rice, paddy     39
Rubber, natural     39

                              column  null_count
                                Crop           0
            Precipitation (mm

## 2. EDA — achados

            O rendimento varia por cultura (escalas diferentes de cacau, café, etc.),
            então `Crop` entra como one-hot. A precipitação e as umidades são as
            variáveis climáticas mais associadas ao `Yield`.

            ```
                                             column  count      mean       std      min      25%       50%       75%        max
0              Precipitation (mm day-1)  156.0   2486.50    289.46  1934.62  2302.99   2424.55   2718.08    3085.79
1  Specific Humidity at 2 Meters (g/kg)  156.0     18.20      0.29    17.54    18.03     18.27     18.40      18.70
2     Relative Humidity at 2 Meters (%)  156.0     84.74      1.00    82.11    84.12     84.85     85.51      86.10
3           Temperature at 2 Meters (C)  156.0     26.18      0.26    25.56    26.02     26.13     26.30      26.81
4                                 Yield  156.0  56153.10  70421.96  5249.00  8327.75  18871.00  67518.75  203399.00
            ```

            Correlação (numéricas + alvo):

            ```
                                                  Precipitation (mm day-1)  Specific Humidity at 2 Meters (g/kg)  Relative Humidity at 2 Meters (%)  Temperature at 2 Meters (C)  Yield
Precipitation (mm day-1)                                 1.000                                 0.488                              0.749                       -0.084  0.019
Specific Humidity at 2 Meters (g/kg)                     0.488                                 1.000                              0.437                        0.699  0.013
Relative Humidity at 2 Meters (%)                        0.749                                 0.437                              1.000                       -0.337  0.000
Temperature at 2 Meters (C)                             -0.084                                 0.699                             -0.337                        1.000  0.013
Yield                                                    0.019                                 0.013                              0.000                        0.013  1.000
            ```

            Figuras: `figures/eda/distributions.png`, `figures/eda/correlation.png`,
            `figures/eda/yield_vs_precipitation.png`.


## 3. Clusterização e outliers

     K-Means escolhe `k` pelo maior silhouette entre 2 e 6
     (**k=6**, silhouette
     0.445717). Isolation Forest marca
     4 cenários discrepantes na EDA.
     Na regressão, o filtro aprende **só no treino**
     (4 removidos de
     117); o teste fica com
     39 linhas intactas.

     ```
      cluster  registros  rendimento_medio  rendimento_mediano  outliers
4         20         60503.850             20481.5         0
3         52         57655.712             19153.5         0
5         16         55710.625             17605.5         0
2         16         55490.625             18800.5         4
0         12         55159.833             20158.0         0
1         40         52764.275             17485.5         0
     ```


In [1]:
from vinicius_analysis import run_analysis

cluster = run_analysis(random_state=42)
print(cluster.cluster_summary.round(3).to_string(index=False))
print(cluster.outlier_summary)


 cluster  registros  rendimento_medio  rendimento_mediano  outliers
       4         20         60503.850             20481.5         0
       3         52         57655.712             19153.5         0
       5         16         55710.625             17605.5         0
       2         16         55490.625             18800.5         4
       0         12         55159.833             20158.0         0
       1         40         52764.275             17485.5         0
{'selected_k': 6, 'silhouette': 0.445717, 'candidate_scores': [{'k': 2, 'silhouette': 0.3924128249740136}, {'k': 3, 'silhouette': 0.43534616694387185}, {'k': 4, 'silhouette': 0.43967757757993314}, {'k': 5, 'silhouette': 0.4281168358569696}, {'k': 6, 'silhouette': 0.44571743189477164}], 'outlier_method': 'IsolationForest', 'contamination': 0.03, 'outlier_count': 4, 'outlier_rate': 0.025641, 'train_rows': 117, 'train_inliers': 113, 'train_outliers_removed': 4, 'test_rows_untouched': 39}


## 4. Cinco regressões

Mesmo split, mesmo pré-processamento (StandardScaler + OneHot de `Crop`)
e o mesmo filtro de outlier no treino.

| # | Algoritmo | Parte |
|---|---|---|
| 1 | Linear Regression | Higor |
| 2 | Decision Tree | Humberto |
| 3 | KNN Regressor | Humberto |
| 4 | Random Forest | Vinícius |
| 5 | Gradient Boosting | Vinícius |


In [1]:
from comparison import run_comparison, save_outputs

comparison = run_comparison(random_state=42)
report = save_outputs(comparison)
print(comparison.metrics.round(4).to_string(index=False))
print("Melhor modelo (RMSE):", comparison.best_model_name)


            model        mae       rmse     r2   mape
Linear Regression  4267.7089  7852.8201 0.9865 0.1519
Gradient Boosting  4501.7275  9327.5482 0.9810 0.0915
    Random Forest  4424.7363  9486.6105 0.9803 0.0866
    Decision Tree  5483.5721 11013.5343 0.9735 0.1158
    KNN Regressor 18580.1625 30906.3406 0.7910 0.9727
Melhor modelo (RMSE): Linear Regression


## 5. Conclusões e limites

- Melhor RMSE no teste: **Linear Regression**.
- KNN sofre com a escala e a cardinalidade de culturas (156 linhas).
- Árvores e boosting capturam interações clima × cultura; a regressão
  linear também se beneficia do one-hot de `Crop` neste conjunto pequeno.
- Limite: amostra oficial é curta; métricas não devem ser lidas como
  produção agrícola nacional. Não removemos outlier do teste para não
  maquiar o erro.

Figura comparativa: `figures/comparison/rmse_comparison.png`.

## 6. Entrega 2 — Cloud (resumo)

On-Demand 100%, Linux `t3.micro` (2 vCPU, 1 GiB, até 5 Gbit) + 50 GiB gp3.
**N. Virginia é mais barata** (~US$ 11,59/mês vs ~US$ 19,86 em São Paulo).
Com restrição de dado no exterior e acesso rápido, a região escolhida é
**São Paulo (`sa-east-1`)**: LGPD / residência no Brasil e menor latência.
Figuras e passo a passo da calculadora: README e `document/`.
